# Day 01 — Descriptive statistics cho cohort 200 bệnh nhân

**Slide tham khảo chi tiết:** [day01_reference_pack.zip](_static/slides/day01_reference_pack.zip)  
**Slide gốc đầy đủ 20 buổi:** [day01_day20_slides_pack.zip](_static/slides/day01_day20_slides_pack.zip)

## Mục tiêu bài học

- đọc đúng cấu trúc của cohort demo 200 bệnh nhân
- xác định được biến đích, biến liên tục, biến phân loại
- tạo Table 1 gọn cho toàn cohort và theo nhóm EGFR
- vẽ biểu đồ mô tả để nhìn nhanh phân bố dữ liệu

## Nội dung

Day 01 tương ứng phần mở đầu của báo cáo: cần biết dữ liệu có gì, cohort gồm những ai, tỉ lệ EGFR đột biến là bao nhiêu và các biến lâm sàng/radiomics đang được tổ chức như thế nào. Chỉ khi làm xong bước này mới có thể chuyển sang ML ở Day 02.

## Sản phẩm sau bài học

- `table1_overall.csv`
- `table1_by_egfr.csv`
- `hist_age.png`, `boxplot_tumor_size_by_egfr.png`, `bar_sex.png`
- 3–5 dòng diễn giải ngắn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
plt.rcParams["figure.figsize"] = (6,4)
sns.set_style("whitegrid")

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

GITHUB_USER = "ketnoimaytinh797-dotcom"
REPO_NAME = "EGFR-Radiomics-MiniBootcamp"
BRANCH = "main"


def download_from_github(rel_path: str) -> Path:
    import urllib.request
    url = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{rel_path}"
    out = Path(rel_path).name
    urllib.request.urlretrieve(url, out)
    return Path(out)


def load_csv(rel_path: str) -> pd.DataFrame:
    p = Path(rel_path)
    if p.exists():
        return pd.read_csv(p)
    try:
        p2 = download_from_github(rel_path)
        return pd.read_csv(p2)
    except Exception as e:
        raise FileNotFoundError(f"Không tìm thấy {rel_path}. Hãy chỉnh đúng GITHUB_USER/REPO_NAME hoặc upload file thủ công.") from e


df = load_csv("data/nsclc_egfr_radiomics_simplified.csv")
df.head()

## 1. Nhìn nhanh cấu trúc dữ liệu

Trước khi tính toán, cần trả lời được 3 câu: có bao nhiêu bệnh nhân, bao nhiêu cột, biến đích là cột nào.

In [ ]:
df.shape, df.columns.tolist()

In [ ]:
df.dtypes.astype(str).to_frame("dtype")

## 2. Phân loại nhóm biến

Ở bộ dữ liệu demo này có thể tạm chia thành 3 nhóm: lâm sàng, hình thái khối u, radiomics theo ROI.

In [ ]:

clinical_cols = ["age", "sex", "smoking_status", "histology", "stage", "tp53_mutation", "egfr_mutation"]
size_cols = ["tumor_size_mm", "tumor_volume_cm3"]
radiomics_cols = [c for c in df.columns if c.startswith(("intra_", "ring1_", "ring3_", "ring5_"))]

pd.DataFrame({
    "nhóm": ["lâm sàng", "kích thước/hình thái", "radiomics"],
    "số cột": [len(clinical_cols), len(size_cols), len(radiomics_cols)]
})


## 3. Table 1 tổng quan

Mục tiêu là có một bảng cohort tổng quát: biến liên tục báo cáo trung bình, độ lệch chuẩn hoặc trung vị, IQR; biến phân loại báo cáo số lượng và tỉ lệ.

In [ ]:

summary = {
    "n": len(df),
    "age_mean": round(df["age"].mean(), 1),
    "age_sd": round(df["age"].std(ddof=1), 1),
    "tumor_size_median": round(df["tumor_size_mm"].median(), 1),
    "tumor_size_iqr": round(df["tumor_size_mm"].quantile(0.75) - df["tumor_size_mm"].quantile(0.25), 1),
    "male_%": round((df["sex"] == "M").mean() * 100, 1),
    "egfr_%": round(df["egfr_mutation"].mean() * 100, 1),
}
pd.Series(summary).to_frame("value")


In [ ]:

table1_overall = pd.DataFrame([
    ["n", len(df)],
    ["Age, mean ± SD", f"{df['age'].mean():.1f} ± {df['age'].std(ddof=1):.1f}"],
    ["Tumor size, median [IQR]", f"{df['tumor_size_mm'].median():.1f} [{df['tumor_size_mm'].quantile(0.25):.1f}; {df['tumor_size_mm'].quantile(0.75):.1f}]"],
    ["Tumor volume, median [IQR]", f"{df['tumor_volume_cm3'].median():.1f} [{df['tumor_volume_cm3'].quantile(0.25):.1f}; {df['tumor_volume_cm3'].quantile(0.75):.1f}]"],
    ["Male, n (%)", f"{(df['sex'] == 'M').sum()} ({(df['sex'] == 'M').mean()*100:.1f}%)"],
    ["EGFR mutated, n (%)", f"{df['egfr_mutation'].sum()} ({df['egfr_mutation'].mean()*100:.1f}%)"]
], columns=["Variable", "Overall"])
table1_overall


## 4. So sánh EGFR dương tính và EGFR âm tính

Bảng này rất quan trọng vì nó nối trực tiếp từ mô tả sang suy luận.

In [ ]:

cont_vars = ["age", "tumor_size_mm", "tumor_volume_cm3"]
rows = []
for var in cont_vars:
    g1 = df.loc[df.egfr_mutation == 1, var]
    g0 = df.loc[df.egfr_mutation == 0, var]
    p = stats.mannwhitneyu(g1, g0, alternative="two-sided").pvalue
    rows.append([
        var,
        f"{g1.median():.1f} [{g1.quantile(0.25):.1f}; {g1.quantile(0.75):.1f}]",
        f"{g0.median():.1f} [{g0.quantile(0.25):.1f}; {g0.quantile(0.75):.1f}]",
        round(p, 4)
    ])

for var in ["sex", "smoking_status", "histology", "stage"]:
    tab = pd.crosstab(df[var], df["egfr_mutation"])
    p = stats.chi2_contingency(tab)[1]
    rows.append([var, "xem bảng tần số", "xem bảng tần số", round(p, 4)])

table1_by_egfr = pd.DataFrame(rows, columns=["Variable", "EGFR+", "EGFR-", "p_value"])
table1_by_egfr


In [ ]:
pd.crosstab(df["sex"], df["egfr_mutation"], normalize="columns")

## 5. Trực quan hóa cơ bản

Ba hình tối thiểu nên có trong phần mô tả là phân bố tuổi, phân bố kích thước u, và tỉ lệ EGFR theo nhóm.

In [ ]:

out_dir = Path("outputs/day01")
out_dir.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots()
sns.histplot(data=df, x="age", hue="egfr_mutation", bins=15, kde=True, ax=ax)
ax.set_title("Phân bố tuổi theo EGFR")
fig.tight_layout()
fig.savefig(out_dir / "age_hist_by_egfr.png", dpi=150)
plt.show()


In [ ]:

fig, ax = plt.subplots()
sns.boxplot(data=df, x="egfr_mutation", y="tumor_size_mm", ax=ax)
ax.set_xlabel("EGFR mutation")
ax.set_ylabel("Tumor size (mm)")
ax.set_title("Kích thước u theo nhóm EGFR")
fig.tight_layout()
fig.savefig(out_dir / "tumor_size_boxplot.png", dpi=150)
plt.show()


In [ ]:

fig, ax = plt.subplots()
(df.groupby("stage")["egfr_mutation"].mean()*100).sort_index().plot(kind="bar", ax=ax)
ax.set_ylabel("Tỉ lệ EGFR đột biến (%)")
ax.set_title("Tỉ lệ EGFR theo giai đoạn")
fig.tight_layout()
fig.savefig(out_dir / "egfr_rate_by_stage.png", dpi=150)
plt.show()


## 6. Lưu bảng kết quả

Sau mỗi buổi nên lưu ngay output để dùng lại cho capstone.

In [ ]:

table1_overall.to_csv(out_dir / "table1_overall.csv", index=False)
table1_by_egfr.to_csv(out_dir / "table1_by_egfr.csv", index=False)
list(out_dir.iterdir())


## 7. Mẫu diễn giải kết quả

Có thể bắt đầu từ 4 câu ngắn:

- Cohort demo gồm 200 bệnh nhân NSCLC, trong đó tỉ lệ EGFR đột biến khoảng `xx%`.
- Tuổi trung bình và kích thước u trung vị đã được tóm tắt trong Table 1.
- Ở bước mô tả, nhóm EGFR dương tính và EGFR âm tính có thể khác nhau ở một số biến lâm sàng hoặc kích thước u.
- Các khác biệt này cần được kiểm tra tiếp trong các bước mô hình hóa và pathway analysis.

## 8. Tự kiểm tra

1. Vì sao Day 01 phải làm trước khi chạy ML?  
2. Khi nào nên dùng trung vị thay vì trung bình?  
3. Tại sao nên lưu cả bảng csv và hình png ngay trong ngày học?  
4. Biến đích của bộ dữ liệu này là cột nào?

## Nối sang buổi sau

Day 02 sẽ dùng chính cohort này để tạo mô hình dự đoán EGFR, tính ROC AUC và so sánh intra với ring1, ring3, ring5.